Multi-head Attention Plus Data Loading

In [ ]:
# NBVAL_IGNORE_OUTPUT
from importlib.metadata import version

# 打印当前环境中 torch 的版本号，用于确认后续张量运算所依赖的 PyTorch 版本
print("torch version:", version("torch"))


Data Loader from Chapter 2

In [ ]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# 第2章实现的数据集类：把原始文本切分成 (输入序列, 目标序列) 的样本对
# 目标序列相对输入序列整体右移一位，用于“预测下一个词”的训练目标
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        # 使用 tokenizer 把整段文本编码成 token id 列表（保留 <|endoftext|> 特殊标记）
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        # 用滑动窗口把长 token 序列切成长度为 max_length 的子序列
        # stride 控制窗口每次滑动的步长：stride < max_length 时相邻窗口会有重叠
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]           # 输入：第 i 到 i+max_length-1 个 token
            target_chunk = token_ids[i + 1: i + max_length + 1]  # 目标：整体右移一位，即“下一个 token”
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # 数据集样本总数 = 切分出来的窗口数量
        return len(self.input_ids)

    def __getitem__(self, idx):
        # 按索引返回一对 (输入张量, 目标张量)，形状均为 (max_length,)
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # Initialize the tokenizer
    # 使用 GPT-2 的 BPE 分词器
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    # 构建上面定义的滑动窗口数据集
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    # 用 DataLoader 按 batch_size 打包样本，shuffle 控制是否打乱样本顺序
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return dataloader


# 读取用于演示的小文本文件
with open("small-text-sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

vocab_size = 50257   # GPT-2 词表大小
output_dim = 256     # 词嵌入（embedding）维度
max_len = 1024       # 位置编码支持的最大上下文长度
context_length = max_len


# 词元嵌入层：把 token id 映射为 output_dim 维的向量
token_embedding_layer = nn.Embedding(vocab_size, output_dim)
# 位置嵌入层：为每个位置（0 ~ context_length-1）学习一个 output_dim 维向量，
# 用来给模型提供词序信息（因为注意力机制本身不含位置信息）
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 4  # 本示例中实际使用的序列长度（远小于 max_len，便于演示）
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)


In [ ]:
for batch in dataloader:
    x, y = batch  # x: 输入 token id，形状 (batch_size, max_length)；y: 目标 token id，同形状

    # 查表得到每个 token 的词嵌入，形状变为 (batch_size, max_length, output_dim)
    token_embeddings = token_embedding_layer(x)
    # 位置嵌入只依赖位置索引 0..max_length-1，与 batch 无关，形状为 (max_length, output_dim)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    # 词嵌入 + 位置嵌入（广播相加），得到融合了词义与位置信息的最终输入嵌入
    # 形状：(batch_size, max_length, output_dim)
    input_embeddings = token_embeddings + pos_embeddings

    break  # 只取第一个 batch 用于后续演示，不遍历整个 dataloader


In [ ]:
# 打印输入嵌入的形状：(batch_size, max_length, output_dim)
print(input_embeddings.shape)


Multi-head Attention from Chapter 3
Variant A: Simple implementation

In [ ]:
# ===== 多头注意力 变体A：简单实现 =====
# 先实现单头的“带因果掩码的自注意力”，再用多个独立的单头模块拼接（cat）成多头
class CausalSelfAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        # Q、K、V 三个线性投影层，把输入从 d_in 维映射到 d_out 维
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New  # 对注意力权重做 dropout，缓解过拟合
        # register_buffer：注册一个不参与梯度更新、但会随模型一起保存/搬设备的张量
        # torch.triu(..., diagonal=1) 生成主对角线以上为 1（不含对角线）的矩阵，
        # 即“未来位置”的掩码，用来实现因果（causal）注意力——每个位置只能看到自己及之前的位置
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, n_tokens, d_in = x.shape # New batch dimension b  # x 形状: (batch, 序列长度, d_in)
        keys = self.W_key(x)      # (b, n_tokens, d_out)
        queries = self.W_query(x) # (b, n_tokens, d_out)
        values = self.W_value(x)  # (b, n_tokens, d_out)

        # 注意力分数 = Q @ K^T，对最后两维做转置（保留 batch 维不变）
        # 结果形状: (b, n_tokens, n_tokens)，表示每个 query 位置对每个 key 位置的相关度
        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        # 用掩码把“未来位置”对应的分数填成 -inf，softmax 后这些位置的权重会变为 0，
        # 从而保证当前位置只能关注自己和之前的位置（因果性）
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
        # 缩放点积注意力：除以 sqrt(d_k)，防止点积数值过大导致 softmax 梯度过小
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights) # New

        # 用注意力权重对 value 加权求和，得到每个位置的上下文向量
        # 形状: (b, n_tokens, d_out)
        context_vec = attn_weights @ values
        return context_vec


# 多头注意力的“包装器”实现：并行创建 num_heads 个独立的单头注意力模块，
# 每个头拥有自己独立的 W_query/W_key/W_value 参数，互不共享
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # 用 ModuleList 保存 num_heads 个独立的 CausalSelfAttention 头
        self.heads = nn.ModuleList(
            [CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
        # 输出投影层：把拼接后 (num_heads * d_out) 维的向量再线性变换一次
        self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)

    def forward(self, x):
        # 每个头单独计算出形状为 (b, n_tokens, d_out) 的上下文向量，
        # 在最后一维（特征维）拼接（cat），得到 (b, n_tokens, num_heads*d_out)
        # 这是最直观的“多头拆分/合并”方式：多头是并行的独立子模块，最后再拼接
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)


In [ ]:
torch.manual_seed(123)  # 固定随机种子，保证权重初始化可复现

context_length = max_length  # 上下文长度 = 序列长度（本例中为 4）
d_in = output_dim  # 输入特征维度 = 词嵌入维度 256

num_heads=2       # 头数
d_out = d_in // num_heads  # 每个头的输出维度：这里是“先除后拼”，拼接后总维度仍为 d_in

# 变体A：包装器实现，num_heads 个独立头，每个头输出 d_out 维，拼接后总维度为 d_out*num_heads = d_in
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)

batch = input_embeddings
context_vecs = mha(batch)

# 期望形状: (batch_size, max_length, num_heads*d_out) = (8, 4, 256)
print("context_vecs.shape:", context_vecs.shape)


Variant B: Alternative implementation

In [ ]:
# ===== 多头注意力 变体B：更高效的实现 =====
# 与变体A不同，这里不创建多个独立的小模块，而是用一整套 W_query/W_key/W_value
# 一次性把输入投影到完整的 d_out 维，再通过 reshape + transpose 把 d_out 拆分成 (num_heads, head_dim)，
# 从而在张量运算层面并行计算多个头，避免 Python 层面的 for 循环，效率更高
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"  # d_out 必须能被头数整除

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim
        # 每个头实际处理的维度，由 d_out 均分得到

        # 注意：这里 W_query/W_key/W_value 的输出维度直接是完整的 d_out，
        # 而不是像变体A那样每个头单独用一个小的线性层
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs  # 合并多头输出后的投影层
        self.dropout = nn.Dropout(dropout)
        # 因果掩码：与变体A相同，主对角线以上（不含对角线）为 1，表示需要屏蔽的“未来”位置
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        # 多头拆分的关键一步：把最后一维 d_out 拆成 (num_heads, head_dim)，
        # 这只是改变了张量的“视图”（view），并没有真正拷贝出多份独立数据，
        # 数学上等价于把大矩阵按列分块，分给每个头
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        # 把 num_heads 维换到 batch 维之后、num_tokens 维之前，
        # 这样后续矩阵乘法会把 num_heads 当作一个“批量”维度，实现多头并行计算
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        # Q @ K^T，只对最后两维（num_tokens, head_dim）做转置和矩阵乘法：
        # (b, num_heads, num_tokens, head_dim) @ (b, num_heads, head_dim, num_tokens)
        # -> (b, num_heads, num_tokens, num_tokens)，每个头独立得到自己的注意力分数矩阵
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        # 因果掩码：截取到实际序列长度 num_tokens，并转换为布尔类型
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        # 同变体A：把未来位置对应的注意力分数置为 -inf，softmax 后权重变为 0
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # 缩放点积注意力：除以 sqrt(head_dim)（注意这里用的是每个头自己的维度，而非整体 d_out）
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        # 用权重对 value 加权求和后，把 (b, num_heads, num_tokens, head_dim) 转置回
        # (b, num_tokens, num_heads, head_dim)，为下一步重新合并多头做准备
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        # 多头合并：.contiguous() 保证转置后内存连续，才能安全地用 view 重新展平，
        # 把 (num_heads, head_dim) 两维合并回 d_out 维，得到 (b, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection  # 最后再做一次线性投影融合各头信息

        return context_vec


In [ ]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim
d_out = d_in  # 变体B中 d_out 直接等于输入维度，不像变体A那样先除以 num_heads 再拼接

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
context_vecs = mha(batch)

# 期望形状: (batch_size, max_length, d_out) = (8, 4, 256)，与变体A的结果形状一致
print("context_vecs.shape:", context_vecs.shape)
